Compare: model race, tuning, blending
- Race: LGB vs XGB vs CAT
- Optuna tuning for LGB and XGB

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

In [3]:
train = pd.read_csv("train.csv")
y = np.log1p(train["hours_spent"])
X = train.drop(columns=["hours_spent", "log_hours", "hours_bucket", "project_id"])

def add_features(df):
    df = df.copy()
    df["biggest_commit_share"] = df["max_lines_in_one_commit"] / df["lines_added_total"].replace(0, np.nan)
    df["lines_per_active_day"] = df["lines_of_code"] / df["active_commit_days"].replace(0, np.nan)
    df["commits_per_active_day"] = df["num_commits"] / df["active_commit_days"].replace(0, np.nan)
    df["active_day_ratio"] = df["active_commit_days"] / (df["project_span_days"] + 1)
    return df

In [4]:
ORDER_MAPS = {
    "author_prior_projects":     {"0": 0, "1-2": 1, "3-5": 2, "6+": 3},
    "author_account_age_years":  {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3},
}
def encode_ordered(df):
    df = df.copy()
    for col, mapping in ORDER_MAPS.items():
        df[col] = df[col].astype(str).map(mapping)
    return df

In [5]:
def encode_categories(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype("category")
    return df

In [6]:
def encode_yes_no(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        values = set(df[col].dropna().astype(str).unique())
        if values <= {"True", "False"}:
            df[col] = df[col].astype(str).map({"True": 1.0, "False": 0.0})
    return df

In [7]:
def prepare(df):
    df = add_features(df)
    df = encode_ordered(df)
    df = encode_yes_no(df)
    df = encode_categories(df)
    return df

X = prepare(X)

C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_15380\1303781909.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\Fujitsu\AppData\Local\Temp\ipykernel_15380\3138021966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guid

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(X.shape)

(4500, 73)


In [9]:
from sklearn.metrics import root_mean_squared_log_error

def cv_score(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [10]:
bool_cols = X.select_dtypes(include="bool").columns
X = X.astype({c: int for c in bool_cols})


cat_cols = X.select_dtypes(include="category").columns.tolist()
X_cat = X.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(object).fillna("missing").astype(str)

In [11]:
import xgboost as xgb
from catboost import CatBoostRegressor

make_lgb = lambda: lgb.LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1)

make_xgb = lambda: xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    enable_categorical=True, tree_method="hist", random_state=42)

make_cat = lambda: CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=6,
    cat_features=cat_cols, random_seed=42, verbose=0)

print("LightGBM:", round(cv_score(make_lgb, X), 4))
print("XGBoost: ", round(cv_score(make_xgb, X), 4))
print("CatBoost:", round(cv_score(make_cat, X_cat), 4))

LightGBM: 0.6643
XGBoost:  0.6613
CatBoost: 0.6607


In [12]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

d:\Xin Bing\VSC\hour-detective\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
def objective(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 2000),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves = trial.suggest_int("num_leaves", 7, 63),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: lgb.LGBMRegressor(**params, subsample_freq=1, random_state=42, verbose=-1)
    return cv_score(make, X)

In [14]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best score: ", round(study.best_value, 4))
print("Best settings: ", study.best_params)

Best trial: 45. Best value: 0.655465: 100%|██████████| 50/50 [09:16<00:00, 11.13s/it]

Best score:  0.6555
Best settings:  {'n_estimators': 443, 'learning_rate': 0.011845659680012815, 'num_leaves': 59, 'min_child_samples': 29, 'subsample': 0.8149873735971158, 'colsample_bytree': 0.5673542107215191, 'reg_lambda': 0.016908325382149595}


In [15]:
def cv_oof(make_model, X_data):
    oof = np.zeros(len(X_data))
    for train_idx, test_idx in kf.split(X_data):
        model = make_model()
        model.fit(X_data.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] = model.predict(X_data.iloc[test_idx])
    return oof

In [16]:
def score_from_oof(oof):
    pred = np.expm1(oof).clip(0, 140.7)
    return root_mean_squared_log_error(train["hours_spent"], pred)

In [17]:
best_params = study.best_params
make_lgb_best = lambda: lgb.LGBMRegressor(**best_params, subsample_freq=1, random_state=42, verbose=-1)

oof_lgb = cv_oof(make_lgb_best, X)
oof_xgb = cv_oof(make_xgb, X)
oof_cat = cv_oof(make_cat, X_cat)

print("LightGBM: ", round(score_from_oof(oof_lgb), 4))
print("XGBoost: ", round(score_from_oof(oof_xgb), 4))
print("CatBoost: ", round(score_from_oof(oof_cat), 4))

blend = (oof_lgb + oof_xgb + oof_cat) / 3
print("Average of all 3:", round(score_from_oof(blend), 4))

LightGBM:  0.6555
XGBoost:  0.6613
CatBoost:  0.6607
Average of all 3: 0.6526


In [18]:
pred = pd.read_csv("predictions.csv")
print(pred.shape)
print(pred.columns.tolist())
print(pred["hours_spent"].isna().sum())
print(pred["hours_spent"].describe())

(500, 2)
['project_id', 'hours_spent']
0
count    500.000000
mean       9.889884
std       11.748415
min        0.667971
25%        3.385372
50%        5.707498
75%       11.775855
max       93.505477
Name: hours_spent, dtype: float64


In [19]:
import xgboost as xgb
from catboost import CatBoostRegressor

BEST_LGB = {
    "n_estimators": 865, "learning_rate": 0.012964449902814406, "num_leaves": 35,
    "min_child_samples": 11, "subsample": 0.870297694890708,
    "colsample_bytree": 0.5828255385942684, "reg_lambda": 0.002402089003977252,
}

In [20]:
SEEDS = [42, 1, 2]

def make_lgb_seed(seed):
    return lgb.LGBMRegressor(**BEST_LGB, subsample_freq=1, random_state=seed, verbose=-1)

def make_xgb_seed(seed):
    return xgb.XGBRegressor(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        enable_categorical=True, tree_method="hist", random_state=seed
    )

def make_cat_seed(seed):
    return CatBoostRegressor(
        iterations=600, learning_rate=0.06, depth=6,
        cat_features=cat_cols, random_seed=seed, thread_count=-1,
        verbose=0, allow_writing_files=False
    )

def seed_avg_oof(make_with_seed, X_data):
    all_oofs = []
    for s in SEEDS:
        oof = cv_oof(lambda: make_with_seed(s), X_data)
        all_oofs.append(oof)
        print(f"seed {s}: {score_from_oof(oof):.4f}")
    return np.mean(all_oofs, axis=0)

print("LightGBM"); oof_lgb_s = seed_avg_oof(make_lgb_seed, X)
print("XGBoost"); oof_xgb_s = seed_avg_oof(make_xgb_seed, X)
print("CatBoost"); oof_cat_s = seed_avg_oof(make_cat_seed, X_cat)

print()
print("LightGBM (3 seeds): ", round(score_from_oof(oof_lgb_s), 4))
print("XGBoost (3 seeds): ", round(score_from_oof(oof_xgb_s), 4))
print("CatBoost (3 seeds): ", round(score_from_oof(oof_cat_s), 4))

blend_s = (oof_lgb_s + oof_xgb_s + oof_cat_s) / 3
print("Blend (3 seeds): ", round(score_from_oof(blend_s), 4), " <- compare with 0.6532")

LightGBM
seed 42: 0.6554
seed 1: 0.6569
seed 2: 0.6584
XGBoost
seed 42: 0.6613
seed 1: 0.6611
seed 2: 0.6602
CatBoost
seed 42: 0.6672
seed 1: 0.6638


KeyboardInterrupt: 

In [ ]:
def make_cat_seed_slow(seed):
    return CatBoostRegressor(
        iterations=1500, learning_rate=0.03, depth=6,
        cat_features=cat_cols, random_seed=seed, thread_count=-1,
        verbose=0, allow_writing_files=False
    )

print("CatBoost slow"); oof_cat_slow = seed_avg_oof(make_cat_seed_slow, X_cat)

blend_best = (oof_lgb_s + oof_xgb_s + oof_cat_slow) / 3
print("CatBoost slow (3 seeds): ", round(score_from_oof(oof_cat_slow), 4))
print("Blend (3 seeds, slow Cat): ", round(score_from_oof(blend_best), 4))

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
    params = dict(
        n_estimators = trial.suggest_int("n_estimators", 300, 1500),
        learning_rate = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth = trial.suggest_int("max_depth", 3, 10),
        min_child_weight = trial.suggest_float("min_child_weight", 1, 20, log=True),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.001, 10, log=True),
    )
    make = lambda: xgb.XGBRegressor(**params, enable_categorical=True,
                                    tree_method="hist", random_state=42)
    return cv_score(make, X)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

In [ ]:
print("Old XGBoost: 0.6613")
print("Best score: ", round(study_xgb.best_value, 4))
print("Best settings: ", study_xgb.best_params)

In [ ]:
BEST_XGB = study_xgb.best_params

def make_xgb_tuned_seed(seed):
    return xgb.XGBRegressor(**BEST_XGB, enable_categorical=True,
                            tree_method="hist", random_state=seed)

print("XGBoost tuned"); oof_xgb_t = seed_avg_oof(make_xgb_tuned_seed, X)

blend_t = (oof_lgb_s + oof_xgb_t + oof_cat_s) / 3
print()
print("XGBoost old   (3 seeds): 0.6578")
print("XGBoost tuned (3 seeds):", round(score_from_oof(oof_xgb_t), 4))
print("Blend old     (3 seeds): 0.6536")
print("Blend tuned   (3 seeds):", round(score_from_oof(blend_t), 4))
print()
print("BEST_XGB =", BEST_XGB)

In [ ]:
teams = {
    "XGB tuned alone": oof_xgb_t,
    "LGB + XGB": (oof_lgb_s + oof_xgb_t) / 2,
    "LGB + XGB + Cat (equal)": (oof_lgb_s + oof_xgb_t + oof_cat_s) / 3,
    "0.5 XGB + 0.3 LGB + 0.2 Cat": 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_s,
}

for name, oof in teams.items():
    print(f"{name:.30s} {score_from_oof(oof):.4f}")

In [ ]:
best_oof = 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_s

err = pd.DataFrame({
    "real_hours": train["hours_spent"],
    "pred_hours": np.expm1(best_oof).clip(0, 140.7).round(1),
})

err["error"] = best_oof - y
err["times_off"] = np.exp(err["error"].abs()).round(1)

show = ["real_hours", "pred_hours", "times_off",
        "active_commit_days", "project_span_days", "num_commits",
        "lines_of_code", "pct_code_in_first_commit", "readme_word_count",
        "project_category", "url_is_live"]

info = pd.concat([err, train.drop(columns=["hours_spent"])], axis=1)

print("Guessed Too High (real hours were small): ")
display(info.sort_values("error", ascending=False)[show].head(10))

print("Guessed Too Low (real hours were big): ")
display(info.sort_values("error")[show].head(10))

In [ ]:
low_cut, high_cut = err["error"].quantile([0.10,0.90])
group = np.where(err["error"] >= high_cut, "too high",
                 np.where(err["error"] <= low_cut, "too low", "ok"))

num_cols = X.select_dtypes(include="number").columns
Z = (X[num_cols] - X[num_cols].mean()) / X[num_cols].std()

diff = Z.groupby(group).mean().T
diff["high_minus_ok"] = diff["too high"] - diff["ok"]
diff["low_minus_ok"] = diff["too low"] - diff["ok"]

print("🔼 What makes 'too high' projects different:")
print(diff["high_minus_ok"].sort_values(key=abs, ascending=False).head(10).round(2))
print()
print("🔽 What makes 'too low' projects different:")
print(diff["low_minus_ok"].sort_values(key=abs, ascending=False).head(10).round(2))
print()
print("Project types in each group (share of projects):")
display(pd.crosstab(group, train["project_category"], normalize="index").round(2))

In [ ]:
high = group == "too high"
low = group == "too low"

d = (Z[high].mean() - Z[low].mean()).sort_values(key=abs, ascending=False)
print("Positive = more in 'too high', negative = more in 'too low':")
print(d.head(15).round(2))
print()

for col in ["hosting_type", "primary_language", "framework", "commit_cadence"]:
    tab = pd.crosstab(group[high | low], train.loc[high | low, col], normalize="index")
    top = tab.loc["too low"].sub(tab.loc["too high"]).abs().sort_values(ascending=False).head(5).index
    print(f"\n{col} (share in each group):")
    print(tab[top].round(2))

In [ ]:
np.save("oof_lgb_s.npy", oof_lgb_s)
np.save("oof_xgb_t.npy", oof_xgb_t)
np.save("oof_cat_s.npy", oof_cat_s)
print("Saved!")

In [21]:
oof_lgb_s = np.load("oof_lgb_s.npy")
oof_xgb_t = np.load("oof_xgb_t.npy")
oof_cat_s = np.load("oof_cat_s.npy")

In [ ]:
def objective_cat(trial):
    params = dict(
        iterations = trial.suggest_int("iterations", 300, 1000),
        learning_rate = trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        depth = trial.suggest_int("depth", 4, 8),
        l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
        random_strength = trial.suggest_float("random_strength", 0.1, 5, log=True),
    )
    make = lambda: CatBoostRegressor(**params, cat_features=cat_cols, random_seed=42,
                                     thread_count=-1, verbose=0, allow_writing_files=False)
    return cv_score(make, X_cat)

study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=20, timeout=3600, show_progress_bar=True)

print("Old CatBoost (fast, seed 42): 0.6672")
print("Best score: ", round(study_cat.best_value, 4))
print("Best settings: ", study_cat.best_params)

In [ ]:
BEST_CAT = study_cat.best_params

def make_cat_tuned_seed(seed):
    return CatBoostRegressor(**BEST_CAT, cat_features=cat_cols, random_seed=seed,
                             thread_count=-1, verbose=0, allow_writing_files=False)
print("CatBoost tuned"); oof_cat_t = seed_avg_oof(make_cat_tuned_seed, X_cat)
np.save("oof_cat_t.npy", oof_cat_t)

print()
print("CatBoost old (3 seeds): 0.6619")
print("CatBoost tuned (3 seeds): ", round(score_from_oof(oof_cat_t), 4))
print()

In [ ]:
teams = {
    "Current: 0.5 XGB + 0.3 LGB + 0.2 Cat(old)": 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_s,
    "Equal, Cat tuned": (oof_xgb_t + oof_lgb_s + oof_cat_t) / 3,
    "0.5 XGB + 0.3 LGB + 0.2 CAT(tuned)": 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_t,
    "0.4 XGB + 0.2 LGB + 0.4 CAT(tuned)": 0.4 * oof_xgb_t + 0.2 * oof_lgb_s + 0.4 * oof_cat_t,
}

for name, oof in teams.items():
    print(f"{name:45s} {score_from_oof(oof):.4f}")

print()
print("BEST_CAT = ", BEST_CAT)

In [22]:
from sklearn.linear_model import LinearRegression

oof_lgb_s = np.load("oof_lgb_s.npy")
oof_xgb_t = np.load("oof_xgb_t.npy")
oof_cat_t = np.load("oof_cat_t.npy")

best_oof = 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_t

stretched = np.zeros(len(y))
for tr, te in kf.split(best_oof):
    lr = LinearRegression().fit(best_oof[tr].reshape(-1, 1), y.iloc[tr])
    stretched[te] = lr.predict(best_oof[te].reshape(-1, 1))

print("Shipped blend:          ", round(score_from_oof(best_oof), 4))
print("Shipped blend, stretched:", round(score_from_oof(stretched), 4))

lr_all = LinearRegression().fit(best_oof.reshape(-1, 1), y)
print(f"\nStretch: new = {lr_all.intercept_:.4f} + {lr_all.coef_[0]:.4f} x old")

Shipped blend:           0.6519
Shipped blend, stretched: 0.6531

Stretch: new = -0.0660 + 1.0348 x old


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.base import clone

num_cols = X_cat.select_dtypes(include="number").columns.tolist()
log_cols = [c for c in num_cols if X_cat[c].min() >= 0 and X_cat[c].max() > 50]
plain_cols = [c for c in num_cols if c not in log_cols]

log_pipe = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("fill", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

plain_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])

onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False)

prep = ColumnTransformer([
    ("log", log_pipe, log_cols),
    ("plain", plain_pipe, plain_cols),
    ("cat", onehot, cat_cols),
])

test_out = clone(prep).fit_transform(X_cat)
print("Big counts (log): ", len(log_cols))
print("Other numbers: ", len(plain_cols))
print("Word columns: ", len(cat_cols))
print("Shape after cleaning: ", test_out.shape)
print("Any empty cells left? ", np.isnan(test_out).any())

In [ ]:
from sklearn.neural_network import MLPRegressor

def make_nn(seed=42):
    return Pipeline([
        ("prep", clone(prep)),
        ("nn", MLPRegressor(
            hidden_layer_sizes=(64,32),
            alpha=1e-3,
            learning_rate_init=1e-3,
            batch_size=64,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=seed,
        )),
    ])

oof_nn = cv_oof(make_nn, X_cat)
np.save("oof_nn.npy", oof_nn)

print("Neural network alone: ", round(score_from_oof(oof_nn), 4))
print("(Trees alone were about 0.6525 - 0.6575)")

In [23]:
oof_lgb_s = np.load("oof_lgb_s.npy")
oof_xgb_t = np.load("oof_xgb_t.npy")
oof_cat_t = np.load("oof_cat_t.npy")
oof_nn    = np.load("oof_nn.npy")

trees = 0.5 * oof_xgb_t + 0.3 * oof_lgb_s + 0.2 * oof_cat_t
print(f"Trees team (shipped): {score_from_oof(trees):.4f}", )

for w in [0.05, 0.10, 0.15, 0.20]:
    mix = (1 - w) * trees + w * oof_nn
    print(f"Trees + {w:.0%} neural net: {score_from_oof(mix):.4f}")

err_xgb, err_lgb = oof_xgb_t - y, oof_lgb_s - y
err_trees, err_nn = trees - y, oof_nn - y
print()
print(f"Mistake similarity, XGB vs LGB:        {np.corrcoef(err_xgb, err_lgb)[0, 1]:.2f}")
print(f"Mistake similarity, trees vs neural net: {np.corrcoef(err_trees, err_nn)[0, 1]:.2f}")

Trees team (shipped): 0.6519
Trees + 5% neural net: 0.6519
Trees + 10% neural net: 0.6524
Trees + 15% neural net: 0.6533
Trees + 20% neural net: 0.6547

Mistake similarity, XGB vs LGB:        0.99
Mistake similarity, trees vs neural net: 0.89


In [26]:
BEST_LGB = {"n_estimators": 865, "learning_rate": 0.012964449902814406, "num_leaves": 35,
            "min_child_samples": 11, "subsample": 0.870297694890708,
            "colsample_bytree": 0.5828255385942684, "reg_lambda": 0.002402089003977252}
BEST_XGB = {"n_estimators": 814, "learning_rate": 0.010311069581219252, "max_depth": 9,
            "min_child_weight": 7.614778905496447, "subsample": 0.5864764800801583,
            "colsample_bytree": 0.3503811143222532, "reg_lambda": 0.024691999110864553}
BEST_CAT = {"iterations": 849, "learning_rate": 0.037576200996845, "depth": 8,
            "l2_leaf_reg": 2.7913342621237325, "random_strength": 1.100700356059648}

LGB_CAREFUL = {**BEST_LGB, "num_leaves": 15, "min_child_samples": 40, "reg_lambda": 5.0}
XGB_CAREFUL = {**BEST_XGB, "max_depth": 5, "min_child_weight": 20, "reg_lambda": 5.0}
CAT_CAREFUL = {**BEST_CAT, "depth": 6, "l2_leaf_reg": 10}

oof_lgb_c = cv_oof(lambda: lgb.LGBMRegressor(**LGB_CAREFUL, subsample_freq=1,
                                             random_state=42, verbose=-1), X)
oof_xgb_c = cv_oof(lambda: xgb.XGBRegressor(**XGB_CAREFUL, enable_categorical=True,
                                            tree_method="hist", random_state=42), X)
oof_cat_c = cv_oof(lambda: CatBoostRegressor(**CAT_CAREFUL, cat_features=cat_cols,
                                             random_seed=42, thread_count=-1,
                                             verbose=0, allow_writing_files=False), X_cat)

blend_c = 0.5 * oof_xgb_c + 0.3 * oof_lgb_c + 0.2 * oof_cat_c
np.save("blend_careful.npy", blend_c)

print("Careful LightGBM:", round(score_from_oof(oof_lgb_c), 4), "  (was 0.6554)")
print("Careful XGBoost: ", round(score_from_oof(oof_xgb_c), 4), "  (was 0.6536)")
print("Careful CatBoost:", round(score_from_oof(oof_cat_c), 4), "  (was 0.6594)")
print("Careful team:    ", round(score_from_oof(blend_c), 4), "  (shipped team: 0.6519)")

Careful LightGBM: 0.6585   (was 0.6554)
Careful XGBoost:  0.6587   (was 0.6536)
Careful CatBoost: 0.6639   (was 0.6594)
Careful team:     0.6569   (shipped team: 0.6519)
